# Reconstrução completa da análise VADER do TCC

Este notebook:

1. coleta comentários históricos de `r/programming`, `r/cscareerquestions` e `r/devops`;
2. usa o período de 2021 a 2024;
3. seleciona até 2.000 comentários por comunidade;
4. aplica o VADER;
5. gera tabelas, gráficos e arquivos para atualizar o GitHub;
6. separa a **base completa privada** dos arquivos adequados para um repositório público.

> A execução é uma reconstrução posterior da metodologia. Ela não garante os mesmos comentários da amostra original que não foi preservada.

In [ ]:
!pip -q install pandas requests nltk matplotlib langdetect python-dateutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 10.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
from __future__ import annotations

import hashlib
import json
import random
import re
import time
import zipfile
from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import nltk
import pandas as pd
import requests
from langdetect import DetectorFactory, LangDetectException, detect
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download("vader_lexicon", quiet=True)
DetectorFactory.seed = 2026

BASE = Path("/content/vader_tcc_completo")
PUBLICO = BASE / "arquivos_para_github"
PRIVADO = BASE / "base_completa_privada"
PUBLICO.mkdir(parents=True, exist_ok=True)
PRIVADO.mkdir(parents=True, exist_ok=True)

API_URL = "https://arctic-shift.photon-reddit.com/api/comments/search"
SUBREDDITS = ["programming", "cscareerquestions", "devops"]
DATA_INICIAL = datetime(2021, 1, 1, tzinfo=timezone.utc)
DATA_FINAL = datetime(2025, 1, 1, tzinfo=timezone.utc)  # exclusivo
META_POR_SUBREDDIT = 2000
SEMENTE = 2026
MIN_CARACTERES = 40
PAUSA = 0.55
TENTATIVAS = 5

TERMOS_ESTRESSE = [
    "burnout", "burning out", "stress", "stressed", "stressful",
    "deadline", "deadlines", "overworked", "workload", "overtime",
    "tired", "exhausted", "exhausting", "exhaustion", "drained",
    "anxious", "anxiety", "pressure", "on call", "on-call",
    "incident", "production issue", "production issues", "outage",
    "bug", "crash", "overload", "can't keep up", "cannot keep up"
]

print("Configuração carregada.")

Configuração carregada.


## 1. Coleta

A coleta usa janelas semanais. Para diminuir viés temporal, a ordenação alterna entre crescente e decrescente. O programa não salva nomes de usuários.

In [ ]:
def extrair_lista(payload: Any) -> list[dict[str, Any]]:
    if isinstance(payload, list):
        return [x for x in payload if isinstance(x, dict)]
    if isinstance(payload, dict):
        for chave in ("data", "results", "comments"):
            valor = payload.get(chave)
            if isinstance(valor, list):
                return [x for x in valor if isinstance(x, dict)]
    return []


def requisitar(
    session: requests.Session,
    subreddit: str,
    inicio: datetime,
    fim: datetime,
    ordem: str,
) -> list[dict[str, Any]]:
    params = {
        "subreddit": subreddit,
        "after": inicio.date().isoformat(),
        "before": fim.date().isoformat(),
        "limit": 100,
        "sort": ordem,
        "fields": "id,subreddit,created_utc,body",
    }

    ultimo_erro = None
    for tentativa in range(1, TENTATIVAS + 1):
        try:
            resposta = session.get(API_URL, params=params, timeout=90)

            if resposta.status_code == 429:
                espera = min(60, 5 * tentativa)
                print(f"Limite temporário. Aguardando {espera}s...")
                time.sleep(espera)
                continue

            resposta.raise_for_status()
            payload = resposta.json()

            if isinstance(payload, dict):
                mensagem = str(payload.get("error") or payload.get("message") or "")
                if "timed out" in mensagem.lower():
                    raise RuntimeError(mensagem)

            return extrair_lista(payload)

        except Exception as erro:
            ultimo_erro = erro
            espera = min(30, 2 ** tentativa)
            time.sleep(espera)

    print(f"Janela ignorada após falhas: r/{subreddit}, {inicio.date()} — {ultimo_erro}")
    return []


def idioma_ingles(texto: str) -> bool:
    try:
        return detect(texto) == "en"
    except LangDetectException:
        return False


def normalizar_texto(texto: str) -> str:
    texto = texto.lower()
    texto = re.sub(r"https?://\\S+|www\\.\\S+", " ", texto)
    texto = re.sub(r"\\s+", " ", texto)
    return texto.strip()


def localizar_termos(texto: str) -> list[str]:
    texto = normalizar_texto(texto)
    encontrados = []
    for termo in TERMOS_ESTRESSE:
        padrao = r"(?<!\\w)" + re.escape(termo) + r"(?!\\w)"
        if re.search(padrao, texto):
            encontrados.append(termo)
    return list(dict.fromkeys(encontrados))


def data_iso(valor: Any) -> str:
    try:
        epoch = int(float(valor))
        return datetime.fromtimestamp(epoch, tz=timezone.utc).isoformat()
    except Exception:
        return str(valor or "").strip()


def preparar(item: dict[str, Any]) -> dict[str, Any] | None:
    corpo = str(item.get("body") or "").strip()
    if not corpo or corpo in {"[deleted]", "[removed]"}:
        return None
    if len(corpo) < MIN_CARACTERES:
        return None
    if not idioma_ingles(corpo):
        return None

    termos = localizar_termos(corpo)
    if not termos:
        return None

    identificador = str(item.get("id") or corpo)
    return {
        "comment_id_hash": hashlib.sha256(identificador.encode("utf-8")).hexdigest()[:16],
        "subreddit": str(item.get("subreddit") or "").lower(),
        "created_utc": data_iso(item.get("created_utc")),
        "body_original": corpo,
        "termos_estresse": "; ".join(termos),
        "fonte_reconstrucao": "Arctic Shift API",
    }


def janelas_semanais(inicio: datetime, fim: datetime):
    atual = inicio
    indice = 0
    while atual < fim:
        proxima = min(atual + timedelta(days=7), fim)
        yield indice, atual, proxima
        atual = proxima
        indice += 1


def coletar_candidatos() -> pd.DataFrame:
    session = requests.Session()
    session.headers.update({
        "User-Agent": "PesquisaAcademicaVADER/1.0 (uso educacional)"
    })

    registros: dict[str, dict[str, Any]] = {}

    for subreddit in SUBREDDITS:
        print(f"\\nColetando r/{subreddit}...")
        total_janelas = 0

        for indice, inicio, fim in janelas_semanais(DATA_INICIAL, DATA_FINAL):
            ordem = "asc" if indice % 2 == 0 else "desc"
            itens = requisitar(session, subreddit, inicio, fim, ordem)

            for item in itens:
                registro = preparar(item)
                if registro:
                    registros[registro["comment_id_hash"]] = registro

            total_janelas += 1
            if total_janelas % 25 == 0:
                atual_sub = sum(
                    1 for r in registros.values()
                    if r["subreddit"] == subreddit
                )
                print(f"  {total_janelas} semanas processadas; {atual_sub} candidatos válidos.")

            time.sleep(PAUSA)

    df = pd.DataFrame(registros.values())
    if df.empty:
        raise RuntimeError("Nenhum comentário válido foi coletado.")
    return df


candidatos = coletar_candidatos()
print("\\nCandidatos válidos por comunidade:")
print(candidatos.groupby("subreddit").size())

\nColetando r/programming...
  25 semanas processadas; 82 candidatos válidos.
  50 semanas processadas; 202 candidatos válidos.
  75 semanas processadas; 278 candidatos válidos.
  100 semanas processadas; 360 candidatos válidos.
  125 semanas processadas; 444 candidatos válidos.
  150 semanas processadas; 516 candidatos válidos.
  175 semanas processadas; 622 candidatos válidos.
  200 semanas processadas; 698 candidatos válidos.
\nColetando r/cscareerquestions...
  25 semanas processadas; 96 candidatos válidos.
  50 semanas processadas; 166 candidatos válidos.
  75 semanas processadas; 246 candidatos válidos.
  100 semanas processadas; 321 candidatos válidos.
  125 semanas processadas; 423 candidatos válidos.
  150 semanas processadas; 518 candidatos válidos.
  175 semanas processadas; 592 candidatos válidos.
  200 semanas processadas; 663 candidatos válidos.
\nColetando r/devops...
  25 semanas processadas; 79 candidatos válidos.
  50 semanas processadas; 186 candidatos válidos.
  75 

## 2. Seleção da amostra e aplicação do VADER

In [ ]:
def selecionar_amostra(df: pd.DataFrame) -> pd.DataFrame:
    partes = []
    for indice, subreddit in enumerate(SUBREDDITS):
        grupo = df[df["subreddit"] == subreddit].drop_duplicates("comment_id_hash")
        n = min(META_POR_SUBREDDIT, len(grupo))
        if n:
            partes.append(grupo.sample(n=n, random_state=SEMENTE + indice))
        print(f"r/{subreddit}: {len(grupo)} candidatos; {n} selecionados.")

    amostra = pd.concat(partes, ignore_index=True)
    return amostra.sort_values(["subreddit", "created_utc"]).reset_index(drop=True)


def classificar(compound: float) -> str:
    if compound >= 0.05:
        return "positivo"
    if compound <= -0.05:
        return "negativo"
    return "neutro"


amostra = selecionar_amostra(candidatos)
analisador = SentimentIntensityAnalyzer()

pontos = amostra["body_original"].apply(analisador.polarity_scores).apply(pd.Series)
for coluna in ["neg", "neu", "pos", "compound"]:
    amostra[coluna] = pontos[coluna].astype(float)

amostra["classe_sentimento"] = amostra["compound"].apply(classificar)

print("\\nAmostra final:")
print(amostra.groupby("subreddit").size())
print(f"Total: {len(amostra)}")

if len(amostra) < META_POR_SUBREDDIT * len(SUBREDDITS):
    print(
        "\\nATENÇÃO: o total ficou abaixo de 6.000. "
        "Não complete artificialmente; informe o número efetivamente obtido."
    )

r/programming: 733 candidatos; 733 selecionados.
r/cscareerquestions: 696 candidatos; 696 selecionados.
r/devops: 866 candidatos; 866 selecionados.
\nAmostra final:
subreddit
cscareerquestions    696
devops               866
programming          733
dtype: int64
Total: 2295
\nATENÇÃO: o total ficou abaixo de 6.000. Não complete artificialmente; informe o número efetivamente obtido.


## 3. Geração dos arquivos

In [ ]:
# Base completa para guardar de forma privada.
amostra.to_csv(
    PRIVADO / "amostra_completa_privada.csv",
    index=False,
    encoding="utf-8-sig",
)

# Versão pública sem o texto integral.
colunas_publicas = [
    "comment_id_hash", "subreddit", "created_utc",
    "termos_estresse", "neg", "neu", "pos",
    "compound", "classe_sentimento", "fonte_reconstrucao"
]
amostra[colunas_publicas].to_csv(
    PUBLICO / "resultados_vader.csv",
    index=False,
    encoding="utf-8-sig",
)

# Resumo por comunidade e total.
resumo = (
    amostra.groupby(["subreddit", "classe_sentimento"])
    .size()
    .rename("quantidade")
    .reset_index()
)
resumo["percentual"] = (
    resumo["quantidade"]
    / resumo.groupby("subreddit")["quantidade"].transform("sum")
    * 100
).round(2)

total = (
    amostra.groupby("classe_sentimento")
    .size()
    .rename("quantidade")
    .reset_index()
)
total["subreddit"] = "TOTAL"
total["percentual"] = (total["quantidade"] / len(amostra) * 100).round(2)

resumo_final = pd.concat([resumo, total], ignore_index=True)
resumo_final.to_csv(
    PUBLICO / "resumo_sentimentos.csv",
    index=False,
    encoding="utf-8-sig",
)

# Frequência dos termos.
contador = Counter()
for celula in amostra["termos_estresse"].fillna("").astype(str):
    for termo in [x.strip() for x in celula.split(";") if x.strip()]:
        contador[termo] += 1

frequencia = pd.DataFrame(
    contador.most_common(),
    columns=["termo", "quantidade_de_comentarios"],
)
frequencia["percentual_da_amostra"] = (
    frequencia["quantidade_de_comentarios"] / len(amostra) * 100
).round(2)
frequencia.to_csv(
    PUBLICO / "frequencia_termos.csv",
    index=False,
    encoding="utf-8-sig",
)

# Exemplos anonimizados: apenas trechos curtos, sem usuário.
exemplos = (
    amostra.sort_values("compound")
    .groupby("subreddit", group_keys=False)
    .head(5)
    .copy()
)
exemplos["trecho_anonimizado"] = exemplos["body_original"].str.slice(0, 220)
exemplos[
    [
        "subreddit", "created_utc", "trecho_anonimizado",
        "neg", "neu", "pos", "compound",
        "classe_sentimento", "termos_estresse"
    ]
].to_csv(
    PRIVADO / "exemplos_anonimizados_para_conferencia.csv",
    index=False,
    encoding="utf-8-sig",
)

# Gráfico de sentimentos.
ordem = ["negativo", "neutro", "positivo"]
distribuicao = (
    total.set_index("classe_sentimento")["percentual"]
    .reindex(ordem)
    .fillna(0)
)
plt.figure(figsize=(8, 5))
distribuicao.plot(kind="bar")
plt.xlabel("Classificação")
plt.ylabel("Percentual")
plt.title("Distribuição dos sentimentos segundo o VADER")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(PUBLICO / "grafico_sentimentos.png", dpi=200)
plt.close()

# Gráfico de termos.
top = frequencia.head(12).sort_values("quantidade_de_comentarios")
plt.figure(figsize=(9, 6))
plt.barh(top["termo"], top["quantidade_de_comentarios"])
plt.xlabel("Quantidade de comentários")
plt.ylabel("Termo")
plt.title("Termos associados ao estresse mais recorrentes")
plt.tight_layout()
plt.savefig(PUBLICO / "grafico_termos.png", dpi=200)
plt.close()

print("Arquivos gerados.")
display(resumo_final)
display(frequencia.head(15))

Arquivos gerados.


,subreddit,classe_sentimento,quantidade,percentual
0,cscareerquestions,negativo,247,35.49
1,cscareerquestions,neutro,38,5.46
2,cscareerquestions,positivo,411,59.05
3,devops,negativo,255,29.45
4,devops,neutro,69,7.97
5,devops,positivo,542,62.59
6,programming,negativo,258,35.20
7,programming,neutro,51,6.96
8,programming,positivo,424,57.84
9,TOTAL,negativo,760,33.12


,termo,quantidade_de_comentarios,percentual_da_amostra
0,bug,758,33.03
1,stress,292,12.72
2,workload,186,8.10
3,pressure,146,6.36
4,on call,140,6.10
5,incident,137,5.97
6,crash,120,5.23
7,tired,116,5.05
8,deadline,112,4.88
9,outage,97,4.23


## 4. Documentação e pacotes finais

In [ ]:
contagens = amostra.groupby("subreddit").size().to_dict()
total_real = len(amostra)

readme = f"""
# Resultados completos da análise VADER

Reconstrução técnica da etapa empírica do TCC sobre estresse em equipes de
desenvolvimento de software.

## Configuração

- Comunidades: r/programming, r/cscareerquestions e r/devops;
- Período: 2021 a 2024;
- Técnica: VADER;
- Total efetivamente analisado: {total_real};
- Distribuição por comunidade: {contagens};
- Fonte da reconstrução posterior: Arctic Shift API;
- Nomes de usuários não foram coletados.

## Arquivos

- `resultados_vader.csv`: identificadores anonimizados, pontuações e classes;
- `resumo_sentimentos.csv`: quantidades e percentuais;
- `frequencia_termos.csv`: frequência verificável dos termos;
- `grafico_sentimentos.png`;
- `grafico_termos.png`.

## Limitação

Esta é uma reconstrução posterior. Como a base original do trabalho não foi
preservada, não é possível garantir que os comentários sejam exatamente os
mesmos que formaram a amostra descrita no TCC.
""".strip()

(PUBLICO / "README_RESULTADOS_COMPLETOS.md").write_text(readme, encoding="utf-8")

metodo = f"""
# Registro da execução

Data da execução: {datetime.now(timezone.utc).isoformat()}

Total analisado: {total_real}

Comunidades e quantidades:
{json.dumps(contagens, ensure_ascii=False, indent=2)}

Critério VADER:
- compound <= -0,05: negativo;
- -0,05 < compound < 0,05: neutro;
- compound >= 0,05: positivo.

Amostra orientada por termos relacionados a estresse ocupacional. A execução
não constitui diagnóstico psicológico.
""".strip()

(PUBLICO / "REGISTRO_DA_EXECUCAO.md").write_text(metodo, encoding="utf-8")

# Criar o código executável para publicar no GitHub.
codigo = Path("/content/coletar_analisar_completo.py")
codigo.write_text(
    "# Código integral disponível no notebook Executar_analise_completa_VADER_Colab.ipynb\\n"
    "# O notebook registra a coleta, seleção, análise e geração das saídas.\\n",
    encoding="utf-8",
)
(PUBLICO / "NOTA_SOBRE_CODIGO.txt").write_text(
    "O notebook usado na execução deve ser publicado junto com estes resultados.",
    encoding="utf-8",
)

# Copiar o próprio notebook para a pasta pública será feito depois do download,
# pois ele foi carregado no Colab pelo usuário.

zip_publico = Path("/content/ATUALIZAR_GITHUB_RESULTADOS_COMPLETOS.zip")
zip_privado = Path("/content/BASE_COMPLETA_PRIVADA_NAO_PUBLICAR.zip")

with zipfile.ZipFile(zip_publico, "w", zipfile.ZIP_DEFLATED) as zf:
    for arquivo in sorted(PUBLICO.iterdir()):
        if arquivo.is_file():
            zf.write(arquivo, arcname=arquivo.name)

with zipfile.ZipFile(zip_privado, "w", zipfile.ZIP_DEFLATED) as zf:
    for arquivo in sorted(PRIVADO.iterdir()):
        if arquivo.is_file():
            zf.write(arquivo, arcname=arquivo.name)

print(f"Pacote para o GitHub: {zip_publico}")
print(f"Base completa privada: {zip_privado}")

Pacote para o GitHub: /content/ATUALIZAR_GITHUB_RESULTADOS_COMPLETOS.zip
Base completa privada: /content/BASE_COMPLETA_PRIVADA_NAO_PUBLICAR.zip


## 5. Baixar os dois pacotes

- O primeiro ZIP é para atualizar o GitHub.
- O segundo contém os textos completos e **não deve ser publicado em repositório público**.

In [ ]:
from google.colab import files

files.download("/content/ATUALIZAR_GITHUB_RESULTADOS_COMPLETOS.zip")
files.download("/content/BASE_COMPLETA_PRIVADA_NAO_PUBLICAR.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>